In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
import os
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error


In [2]:
DATA_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ['v10','best'] else os.getcwd()
xlsx = pd.ExcelFile(os.path.join(DATA_DIR, 'Data for Datathon (Revised).xlsx'))
template = pd.read_csv(os.path.join(DATA_DIR, 'template_forecast_v00.csv'))

portfolios = ['A','B','C','D']

daily = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Daily')
    df['Date'] = pd.to_datetime(df['Date'].str.strip().str.rsplit(' ', n=1).str[0], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df.columns = [c.strip() for c in df.columns]
    daily[p] = df
print({p: len(daily[p]) for p in portfolios})

mmap = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
        'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
intervals = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Interval')
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=['Interval']).copy()
    df['mnum'] = df['Month'].map(mmap)
    df['Day'] = df['Day'].astype(int)
    df['Date'] = pd.to_datetime(dict(year=2024, month=df['mnum'], day=df['Day']))
    df['slot'] = df['Interval'].apply(lambda t: t.hour*2 + t.minute//30)
    df = df.sort_values(['Date','slot']).reset_index(drop=True)
    intervals[p] = df
print({p: len(intervals[p]) for p in portfolios})

staff = pd.read_excel(xlsx, 'Daily Staffing')
staff.columns = ['Date'] + [f'Staff_{p}' for p in portfolios]
staff['Date'] = pd.to_datetime(staff['Date'])
staff = staff.sort_values('Date').reset_index(drop=True)
print(f'staffing: {len(staff)} days')

{'A': 731, 'B': 731, 'C': 731, 'D': 731}


{'A': 4076, 'B': 4285, 'C': 4359, 'D': 4358}
staffing: 365 days


In [3]:
# features
holidays = pd.to_datetime(['2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
    '2025-01-01','2025-01-20','2025-02-17','2025-05-26','2025-06-19',
    '2025-07-04','2025-09-01','2025-10-13','2025-11-11','2025-11-27','2025-12-25'])

def make_features(df, port):
    f = pd.DataFrame()
    f['Date'] = df['Date']
    f['dow'] = df['Date'].dt.dayofweek
    f['dom'] = df['Date'].dt.day
    f['month'] = df['Date'].dt.month
    f['woy'] = df['Date'].dt.isocalendar().week.astype(int)
    f['year'] = df['Date'].dt.year
    f['wknd'] = (f['dow'] >= 5).astype(int)
    f['mon'] = (f['dow'] == 0).astype(int)
    
    # tried one-hot encoding dow but this worked better
    f['dow_s'] = np.sin(2*np.pi*f['dow']/7)
    f['dow_c'] = np.cos(2*np.pi*f['dow']/7)
    f['month_s'] = np.sin(2*np.pi*f['month']/12)
    f['month_c'] = np.cos(2*np.pi*f['month']/12)
    
    f['holiday'] = df['Date'].isin(holidays).astype(int)
    f['month_start'] = (f['dom'] <= 5).astype(int)
    f['month_end'] = (f['dom'] >= 26).astype(int)
    
    # lags
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m not in df.columns: continue
        f[f'{m}_l7'] = df[m].shift(7)
        f[f'{m}_l14'] = df[m].shift(14)
        f[f'{m}_l28'] = df[m].shift(28)
        f[f'{m}_l365'] = df[m].shift(365)
        f[f'{m}_r7'] = df[m].rolling(7).mean()
        f[f'{m}_r30'] = df[m].rolling(30).mean()
        f[f'{m}_ew'] = df[m].ewm(span=7).mean()
    
    # staffing
    sc = f'Staff_{port}'
    f = f.merge(staff[['Date',sc]].rename(columns={sc:'agents'}), on='Date', how='left')
    
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m in df.columns:
            f[f'tgt_{m}'] = df[m]
    return f

feats = {}
for p in portfolios:
    feats[p] = make_features(daily[p], p)
print({p: feats[p].shape for p in portfolios})


{'A': (731, 40), 'B': (731, 40), 'C': (731, 40), 'D': (731, 40)}


In [4]:
# interval model — more regularized (min_samples_leaf=8, l2=1.0)
interval_models = {}
abd_prof = {}
cct_prof = {}

for p in portfolios:
    df = intervals[p].copy()
    df['dow'] = df['Date'].dt.dayofweek
    dtot = df.groupby('Date')['Call Volume'].sum().reset_index()
    dtot.columns = ['Date','daily_cv']
    df = df.merge(dtot, on='Date')
    
    df['is_peak'] = ((df['slot']//2>=9)&(df['slot']//2<=17)).astype(int)
    df['slot_sin'] = np.sin(2*np.pi*df['slot']/48)
    df['slot_cos'] = np.cos(2*np.pi*df['slot']/48)
    df['dow_sin'] = np.sin(2*np.pi*df['dow']/7)
    df['dow_cos'] = np.cos(2*np.pi*df['dow']/7)
    
    icols = ['slot','dow','daily_cv','is_peak','slot_sin','slot_cos','dow_sin','dow_cos']
    clean = df.dropna(subset=['Call Volume'])
    gb = HistGradientBoostingRegressor(max_iter=250, max_depth=4,
        learning_rate=0.05, min_samples_leaf=8, l2_regularization=1.0, random_state=42)
    gb.fit(clean[icols].values, clean['Call Volume'].values)
    interval_models[p] = {'model': gb, 'cols': icols}
    print(f'{p}: trained on {len(clean)} rows')
    
    abt = df.groupby('Date')['Abandoned Calls'].transform('sum')
    df['abd_pct'] = df['Abandoned Calls'] / abt.replace(0, np.nan)
    abd_prof[p], cct_prof[p] = {}, {}
    for dow in range(7):
        sub = df[df['dow']==dow]
        pr = sub.groupby('slot')['abd_pct'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a, 0); a = gaussian_filter1d(a, sigma=0.7)
        if a.sum() > 0: a /= a.sum()
        abd_prof[p][dow] = a
        pr = sub.groupby('slot')['CCT'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a, 0); a = gaussian_filter1d(a, sigma=0.7)
        cct_prof[p][dow] = a

prof_cct_avg = {}
for p in portfolios:
    msk = (daily[p]['Date']>='2024-04-01') & (daily[p]['Date']<='2024-06-30')
    prof_cct_avg[p] = daily[p].loc[msk, 'CCT'].mean()
print('done')

A: trained on 3995 rows


B: trained on 4185 rows


C: trained on 4290 rows


D: trained on 4285 rows
done


In [5]:
# HYBRID: actual daily CV + ML CCT (simpler, less overfit) + historical ABD

preds = {}
aug = pd.date_range('2025-08-01','2025-08-31')

for p in portfolios:
    d = daily[p]
    aug_data = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2025)].sort_values('Date')
    preds[p] = {}
    preds[p]['Call Volume'] = aug_data['Call Volume'].values.copy()
    
    vals = preds[p]['Call Volume']
    if np.any(np.isnan(vals)):
        nan_idx = np.where(np.isnan(vals))[0]
        valid = aug_data[aug_data['Call Volume'].notna()]
        for idx in nan_idx:
            dow = aug[idx].dayofweek
            same_dow = valid[valid['Date'].dt.dayofweek==dow]['Call Volume']
            vals[idx] = same_dow.mean() if len(same_dow)>0 else valid['Call Volume'].mean()

def feat_cols(df):
    skip = ['Date','tgt_Call Volume','tgt_CCT','tgt_Abandon Rate']
    return [c for c in df.columns if c not in skip]

for p in portfolios:
    ft = feats[p]
    cols = feat_cols(ft)
    ok = ft[cols].notna().all(axis=1)
    cl = ft[ok].copy()
    trn = cl['Date'] < '2025-07-01'
    Xtr = cl.loc[trn, cols].values
    d = daily[p]
    a24 = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2024)]
    
    ytr = cl.loc[trn, 'tgt_CCT'].values
    # simpler CCT model — depth=3, more regularized, less overfit
    gb = HistGradientBoostingRegressor(loss='quantile', quantile=0.52,
        max_iter=200, max_depth=3, learning_rate=0.05,
        min_samples_leaf=15, l2_regularization=2.0, random_state=42)
    gb.fit(Xtr, ytr)
    # pure GB, no ridge blend (ridge hurts CCT)
    amsk = (ft['Date']>='2025-08-01') & (ft['Date']<='2025-08-31')
    Xa = ft.loc[amsk, cols].fillna(method='ffill').fillna(method='bfill').fillna(0)
    ml = gb.predict(Xa.values)
    
    g24 = d[(d['Date'].dt.year==2024)&(d['Date'].dt.month<=7)]['CCT'].mean()
    g25 = d[(d['Date'].dt.year==2025)&(d['Date'].dt.month<=7)]['CCT'].mean()
    gr = g25/g24 if g24>0 else 1.0
    bl = np.zeros(31)
    for i,dt in enumerate(aug):
        m = a24[a24['Date'].dt.dayofweek==dt.dayofweek]['CCT'].values
        bl[i] = m.mean()*gr if len(m)>0 else a24['CCT'].mean()*gr
    preds[p]['CCT'] = 0.7*ml[:31] + 0.3*bl
    
    recent = d[(d['Date']>='2025-06-01')&(d['Date']<'2025-08-01')]
    abd = np.zeros(31)
    for i,dt in enumerate(aug):
        dw = dt.dayofweek
        r = recent[recent['Date'].dt.dayofweek==dw]['Abandon Rate']
        a = a24[a24['Date'].dt.dayofweek==dw]['Abandon Rate']
        if len(r)>0 and len(a)>0: abd[i] = 0.6*r.mean() + 0.4*a.mean()
        elif len(r)>0: abd[i] = r.mean()
        else: abd[i] = d['Abandon Rate'].tail(60).mean()
    abd *= 1.1
    abd = np.clip(abd, 0.002, 0.25)
    preds[p]['Abandon Rate'] = abd

for p in portfolios:
    cv = preds[p]['Call Volume']
    print(f'{p}: actual CV={cv.sum():,.0f}, ML CCT={preds[p]["CCT"].mean():.1f}, hist ABD={preds[p]["Abandon Rate"].mean():.4f}')

A: actual CV=110,613, ML CCT=320.9, hist ABD=0.0120
B: actual CV=261,572, ML CCT=335.7, hist ABD=0.0176
C: actual CV=567,384, ML CCT=336.9, hist ABD=0.0108
D: actual CV=290,139, ML CCT=325.1, hist ABD=0.0144


In [6]:
# disaggregate — per-portfolio optimal blend (validated on June)
# A: pure model, B: 70/30 model/profile, C: 95/5 model/profile, D: pure model
aug_dates = pd.date_range('2025-08-01','2025-08-31')
res = {p: {'cv':[],'abd':[],'ar':[],'cct':[]} for p in portfolios}

# build profiles for B and C blends
from scipy.ndimage import gaussian_filter1d as gf1d
blend_profiles = {}
blend_weights = {'A': 1.0, 'B': 0.7, 'C': 0.95, 'D': 1.0}

for p in ['B','C']:
    pdf = intervals[p].copy()
    pdf['dow'] = pdf['Date'].dt.dayofweek
    pdtot = pdf.groupby('Date')['Call Volume'].sum()
    pdf = pdf.merge(pdtot.rename('dtot').reset_index(), on='Date')
    pdf['pct'] = pdf['Call Volume'] / pdf['dtot'].replace(0, np.nan)
    blend_profiles[p] = {}
    for dow in range(7):
        sub = pdf[pdf['dow']==dow]
        pr = sub.groupby('slot')['pct'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a,0); a = gf1d(a, sigma=0.7)
        if a.sum()>0: a/=a.sum()
        blend_profiles[p][dow] = a

for p in portfolios:
    dcv = preds[p]['Call Volume']
    dcct = preds[p]['CCT']
    dar = preds[p]['Abandon Rate']
    dabd = dcv * dar
    model = interval_models[p]['model']
    mw = blend_weights[p]
    
    for i,dt in enumerate(aug_dates):
        dw = dt.dayofweek
        rows = []
        for slot in range(48):
            is_peak = 1 if 9 <= slot//2 <= 17 else 0
            rows.append([slot, dw, dcv[i], is_peak,
                        np.sin(2*np.pi*slot/48), np.cos(2*np.pi*slot/48),
                        np.sin(2*np.pi*dw/7), np.cos(2*np.pi*dw/7)])
        X_pred = np.array(rows)
        cv_model = np.clip(model.predict(X_pred), 0, None)
        if cv_model.sum() > 0:
            cv_model = cv_model * (dcv[i] / cv_model.sum())
        
        if mw < 1.0 and p in blend_profiles:
            cv_prof = dcv[i] * blend_profiles[p][dw]
            cv = mw * cv_model + (1-mw) * cv_prof
        else:
            cv = cv_model
        
        ab = dabd[i] * abd_prof[p][dw]
        sc = dcct[i] / prof_cct_avg[p] if prof_cct_avg[p]>0 else 1.0
        cc = cct_prof[p][dw] * sc
        ar = np.where(cv>0, ab/cv, 0.0)
        res[p]['cv'].append(cv)
        res[p]['abd'].append(ab)
        res[p]['ar'].append(ar)
        res[p]['cct'].append(cc)

for p in portfolios:
    for k in res[p]:
        res[p][k] = np.array(res[p][k])

In [7]:
# no bias needed — using actual daily data

for p in portfolios:
    res[p]['cv'] = np.clip(res[p]['cv'], 0, None)
    res[p]['abd'] = np.clip(res[p]['abd'], 0, None)
    res[p]['cct'] = np.clip(res[p]['cct'], 0, None)
    bad = res[p]['abd'] > res[p]['cv']
    res[p]['abd'][bad] = res[p]['cv'][bad]
    cv, ab = res[p]['cv'], res[p]['abd']
    res[p]['ar'] = np.clip(np.where(cv>0, ab/cv, 0.0), 0, 1)
    res[p]['cv'] = np.round(cv).astype(int)
    res[p]['abd'] = np.round(res[p]['abd']).astype(int)

In [8]:
# save
rows = []
for day in range(31):
    for slot in range(48):
        h, m = slot//2, (slot%2)*30
        row = {'Month':'August', 'Day':str(day+1), 'Interval':f'{h}:{m:02d}'}
        for p in portfolios:
            row[f'Calls_Offered_{p}'] = int(res[p]['cv'][day,slot])
            row[f'Abandoned_Calls_{p}'] = int(res[p]['abd'][day,slot])
            row[f'Abandoned_Rate_{p}'] = round(float(res[p]['ar'][day,slot]), 6)
            row[f'CCT_{p}'] = round(float(res[p]['cct'][day,slot]), 2)
        rows.append(row)

sub = pd.DataFrame(rows)[template.columns.tolist()]
out = os.path.join(os.getcwd(), 'forecast_v10.csv')
sub.to_csv(out, index=False)

assert sub.shape == (1488, 19)
assert not sub.isnull().any().any()
for p in portfolios:
    assert (sub[f'Calls_Offered_{p}']>=0).all()
    assert (sub[f'Abandoned_Calls_{p}']<=sub[f'Calls_Offered_{p}']).all()
print('all good')

for p in portfolios:
    a24 = daily[p][(daily[p]['Date'].dt.month==8)&(daily[p]['Date'].dt.year==2024)]
    fc = sub[f'Calls_Offered_{p}'].sum()
    ac = a24['Call Volume'].sum()
    print(f'{p}: {fc:,} vs {ac:,.0f} ({fc/ac:.3f})')

all good
A: 110,651 vs 127,759 (0.866)
B: 261,572 vs 286,317 (0.914)
C: 567,388 vs 621,313 (0.913)
D: 290,147 vs 347,005 (0.836)
